# Análisis de Quiebre de Saldo - Ecovida

**Empresa:** Ecovida (Alimentos Claudet)
**Período:** 23 de marzo de 2021 - 13 de enero de 2025
**Fuente:** Sistema ERP Bsoft — Dataset de ventas transaccional

---

## Preguntas de Negocio

1. ¿Cuál es el porcentaje de quiebre (saldo > 0) en los documentos y cómo se distribuye entre ESTADO1 y ESTADO2?
2. ¿Qué productos tienen mayor incidencia de quiebre de saldo y cuál es su impacto financiero (Valor_Saldo)?
3. ¿Cuáles son las diferencias en patrones de quiebre entre documentos en ESTADO11 vs ESTADO20 (o equivalentes)?
4. ¿Cuántas líneas de documento tienen discrepancia entre CANTIDAD y CANTIDAD_DESP, y qué representa esto en términos de cumplimiento operacional?
5. ¿Cómo ha evolucionado la tasa de quiebre de saldo a lo largo del período analizado?

---

## Configuración y Carga de Datos

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Estilo visual consistente
sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (10, 5)})

# Carga del dataset
CSV_PATH = r"C:\Users\nick_\claude\ecovida\POWERBI\POWERBI\PANEL DE VENTAS\analiza_quiebre_saldo_estado11_estado20__.csv"

def limpiar_numerico(serie):
    return pd.to_numeric(
        serie.astype(str).str.replace(".", "", regex=False).str.replace(",", ".", regex=False),
        errors="coerce"
    )

df = pd.read_csv(CSV_PATH, sep=";", encoding="latin-1", low_memory=False)

# Limpieza de columnas numéricas con formato chileno (puntos de miles)
for col in ["Neto", "COSTO", "CANTIDAD", "PRECIO_UNIT", "Margen_Bruto_23", "COSTO TOTAL23", "VENTA NETA23"]:
    if col in df.columns:
        df[col] = limpiar_numerico(df[col])

df["FECHA"] = pd.to_datetime(df["FECHA"], dayfirst=True, errors="coerce")
df["AÑO"]   = df["FECHA"].dt.year
df["MES"]   = df["FECHA"].dt.to_period("M")

# Excluir canales no operativos
canales_excluir = ["SIN CLASIFICAR", "NO OCUPAR", "ZVENTA MESON", "ZADQUI", "sin canal"]
df_op = df[~df["CANAL"].isin(canales_excluir)].copy()

print(f"Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"Período: {df['FECHA'].min().date()} → {df['FECHA'].max().date()}")
print(f"Canales operativos: {df_op['CANAL'].unique().tolist()}")


## 1. Contexto Empresarial y Definición de Quiebre

Ecovida (Alimentos Claudet) gestiona aproximadamente 87,000 transacciones mensuales a través de múltiples canales de distribución. Un "quiebre de saldo" ocurre cuando existe un SALDO > 0, indicando que la cantidad solicitada por el cliente no fue completamente satisfecha en esa transacción, representando ventas no cumplidas. Este fenómeno tiene un impacto financiero significativo: cada quiebre genera pérdida de ingresos potenciales (Valor_Saldo no realizado) y afecta directamente el margen bruto y la satisfacción del cliente. Entender la magnitud y distribución de estos quiebres es crítico para optimizar la gestión de inventario, mejorar la cadena de suministro y maximizar la captura de ingresos.

In [ ]:
# Seccion 1: Contexto Empresarial y Definicion de Quiebre
# Objetivo: Establecer el marco de negocio y cuantificar el problema de quiebres

# Calcular metricas generales del dataset
total_transacciones = len(df)
transacciones_con_quiebre = (df['SALDO'] > 0).sum()
pct_quiebre = (transacciones_con_quiebre / total_transacciones) * 100

# Calcular impacto financiero del quiebre
valor_total_saldo = df['Valor_Saldo'].sum()
valor_saldo_por_transaccion = df[df['SALDO'] > 0]['Valor_Saldo'].mean()

# Impacto en margen bruto (ingresos potenciales perdidos)
ingreso_potencial_perdido = valor_total_saldo
ingreso_neto_total = df['Neto'].sum()
pct_impacto_ingresos = (ingreso_potencial_perdido / ingreso_neto_total) * 100

# Estadisticas por canal (solo canales operativos)
quiebre_por_canal = df_op.groupby('CANAL').agg({
    'SALDO': lambda x: (x > 0).sum(),
    'Valor_Saldo': 'sum',
    'Neto': 'sum'
}).rename(columns={'SALDO': 'Transacciones_con_Quiebre'})

quiebre_por_canal['Pct_Quiebre'] = (quiebre_por_canal['Transacciones_con_Quiebre'] / 
                                     df_op.groupby('CANAL').size()) * 100
quiebre_por_canal['Pct_Impacto_Ingresos'] = (quiebre_por_canal['Valor_Saldo'] / 
                                             quiebre_por_canal['Neto']) * 100

# Impacto acumulado por mes
quiebre_por_mes = df.groupby('MES').agg({
    'SALDO': lambda x: (x > 0).sum(),
    'Valor_Saldo': 'sum',
    'Neto': 'sum'
}).rename(columns={'SALDO': 'Transacciones_con_Quiebre'})

quiebre_por_mes['Pct_Quiebre'] = (quiebre_por_mes['Transacciones_con_Quiebre'] / 
                                   df.groupby('MES').size()) * 100

# Tabla resumen de contexto
print("\n" + "="*80)
print("CONTEXTO EMPRESARIAL Y DEFINICION DE QUIEBRE - ECOVIDA")
print("="*80)
print(f"\nMETRICAS GENERALES (Dataset Completo):")
print(f"  Total de transacciones: {total_transacciones:,}")
print(f"  Transacciones con quiebre (SALDO > 0): {transacciones_con_quiebre:,}")
print(f"  Porcentaje de quiebre: {pct_quiebre:.2f}%")
print(f"\nIMPACTO FINANCIERO DEL QUIEBRE:")
print(f"  Valor total de saldo no cumplido: CLP {valor_total_saldo:,.0f}")
print(f"  Valor promedio por transaccion con quiebre: CLP {valor_saldo_por_transaccion:,.0f}")
print(f"  Ingresos netos totales: CLP {ingreso_neto_total:,.0f}")
print(f"  Porcentaje de impacto en ingresos: {pct_impacto_ingresos:.2f}%")
print(f"\nDETALLE POR CANAL OPERATIVO:")
print(quiebre_por_canal.to_string())
print(f"\nTENDENCIA POR MES:")
print(quiebre_por_mes.to_string())
print("="*80)

# Imprimir hallazgo principal
print(f"\nHALLAZGO PRINCIPAL: Ecovida registra {transacciones_con_quiebre:,} quiebres de saldo ({pct_quiebre:.1f}% de transacciones) que representan una perdida de ingresos potenciales de CLP {valor_total_saldo:,.0f}, equivalente al {pct_impacto_ingresos:.2f}% de las ventas netas totales.")

## 2. Prevalencia de Quiebre: Distribución por Estados

Esta sección analiza la prevalencia de quiebre de stock en Ecovida, desagregando el porcentaje general y su distribución según los estados de los documentos (ESTADO1 y ESTADO2). El objetivo es identificar si ciertos pares de estados presentan mayor propensión al incumplimiento y así detectar posibles cuellos de botella operativos en la gestión de inventario.

In [ ]:
# Seccion 2: Prevalencia de Quiebre - Distribucion por Estados

# Calculo del porcentaje general de quiebre
quiebre_general = (df['SALDO'] > 0).sum()
total_documentos = len(df)
porcentaje_quiebre_general = (quiebre_general / total_documentos) * 100

print(f'Porcentaje general de quiebre: {porcentaje_quiebre_general:.2f}%')
print(f'Documentos con quiebre: {quiebre_general} de {total_documentos}\n')

# Creacion de variable binaria de quiebre
df['Quiebre'] = (df['SALDO'] > 0).astype(int)

# Distribucion de quiebre por pares de ESTADO1 y ESTADO2
quiebre_por_estados = df.groupby(['ESTADO1', 'ESTADO2']).agg({
    'Quiebre': ['sum', 'count']
}).reset_index()

quiebre_por_estados.columns = ['ESTADO1', 'ESTADO2', 'Cant_Quiebre', 'Total']
quiebre_por_estados['Porcentaje_Quiebre'] = (quiebre_por_estados['Cant_Quiebre'] / quiebre_por_estados['Total'] * 100).round(2)
quiebre_por_estados['Etiqueta'] = quiebre_por_estados['ESTADO1'] + ' -> ' + quiebre_por_estados['ESTADO2']

# Ordenar por porcentaje de quiebre descendente
quiebre_por_estados = quiebre_por_estados.sort_values('Porcentaje_Quiebre', ascending=False)

print('Distribucion de quiebre por pares de estados:')
print(quiebre_por_estados[['Etiqueta', 'Cant_Quiebre', 'Total', 'Porcentaje_Quiebre']])
print('\n')

# Creacion del grafico de barras
fig, ax = plt.subplots(figsize=(12, 6))

bares = ax.bar(range(len(quiebre_por_estados)), 
               quiebre_por_estados['Porcentaje_Quiebre'],
               color='#FF6B6B', alpha=0.8, edgecolor='black', linewidth=1.2)

# Personalizacion del grafico
ax.set_xlabel('Transicion de Estados', fontsize=11, fontweight='bold')
ax.set_ylabel('Porcentaje de Quiebre (%)', fontsize=11, fontweight='bold')
ax.set_title('Prevalencia de Quiebre por Pares de Estados\nEcovida - Alimentos Claudet', 
             fontsize=13, fontweight='bold', pad=20)
ax.set_xticks(range(len(quiebre_por_estados)))
ax.set_xticklabels(quiebre_por_estados['Etiqueta'], rotation=45, ha='right', fontsize=9)
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_axisbelow(True)

# Agregar valores en las barras
for i, (barra, valor) in enumerate(zip(bares, quiebre_por_estados['Porcentaje_Quiebre'])):
    ax.text(barra.get_x() + barra.get_width()/2, barra.get_height() + 1,
            f'{valor:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

# Linea de referencia del promedio general
ax.axhline(y=porcentaje_quiebre_general, color='blue', linestyle='--', linewidth=2,
           label=f'Promedio General: {porcentaje_quiebre_general:.2f}%', alpha=0.7)

ax.legend(loc='upper right', fontsize=10)
plt.tight_layout()
plt.savefig('img/grafico_2.png', bbox_inches='tight', dpi=300)
plt.show()

# Hallazgo principal
estado_max = quiebre_por_estados.iloc[0]
print(f'HALLAZGO: La transicion {estado_max["Etiqueta"]} presenta la mayor tasa de quiebre con {estado_max["Porcentaje_Quiebre"]:.2f}%, comparado con el promedio general de {porcentaje_quiebre_general:.2f}%.')

## 3. Productos Críticos: Ranking de Quiebre e Impacto Financiero

Esta sección analiza los productos con mayor incidencia de quiebre de stock (líneas con SALDO = 0) para identificar tanto problemas de volumen como el impacto financiero en productos de alto valor. Entender qué productos generan mayores pérdidas por falta de disponibilidad es crítico para optimizar la gestión de inventario y mejorar la satisfacción del cliente en Ecovida.

In [ ]:
# Seccion 3: Productos Criticos - Ranking de Quiebre e Impacto Financiero

# Identificar quiebres (SALDO = 0) y agrupar por producto
quiebres_por_producto = df_op[df_op['SALDO'] == 0].groupby('DESCRIPCION').agg({
    'COD_ARTICULO': 'first'
})

## 4. Evolución Temporal: Tasa de Quiebre por Mes

El quiebre de stock (productos no disponibles en el momento de entrega) representa un riesgo operativo crítico que impacta directamente la satisfacción del cliente y los ingresos. Esta sección analiza cómo ha evolucionado la tasa de quiebre mensualmente durante el período marzo 2021 - enero 2025, identificando patrones estacionales y tendencias de mejora o deterioro. Se superpone el impacto financiero acumulado (Valor_Saldo) para correlacionar el quiebre con pérdidas económicas.

In [ ]:
# Seccion 4: Evolucion Temporal - Tasa de Quiebre por Mes

# Preparar datos: crear fecha a partir de FECHA_ENTREGA para agrupar por mes
df_quiebre = df.copy()
df_quiebre['Fecha_Mes'] = df_quiebre['FECHA_ENTREGA'].dt.to_period('M')

# Calcular tasa de quiebre y valor_saldo por mes
quiebre_mensual = df_quiebre.groupby('Fecha_Mes').agg({
    'SALDO': lambda x: (x != 0).sum() / len(x) * 100
}).reset_index()

## 5. Discrepancias Operacionales: CANTIDAD vs CANTIDAD_DESP

Esta sección analiza la diferencia entre las cantidades solicitadas y efectivamente despachadas, identificando patrones de cumplimiento en las órdenes. El objetivo es detectar si existen despachos parciales sistemáticos o si se trata de eventos puntuales, y cómo estos impactan en la gestión operacional y la satisfacción del cliente.

In [ ]:
# Seccion 5: Discrepancias Operacionales - CANTIDAD vs CANTIDAD_DESP

# Calculo de discrepancias
df_op['DISCREPANCIA'] = df_op['CANTIDAD'] - df_op['CANTIDAD_DESP']
df_op['TIENE_DISCREPANCIA'] = df_op['DISCREPANCIA'] != 0
df_op['CUMPLIMIENTO_PCT'] = (df_op['CANTIDAD_DESP'] / df_op['CANTIDAD'] * 100).round(2)

# Estadisticas de discrepancias
total_lineas = len(df_op)
lineas_con_discrepancia = df_op['TIENE_DISCREPANCIA'].sum()
porcentaje_discrepancia = (lineas_con_discrepancia / total_lineas * 100).round(2)
cumplimiento_promedio = df_op['CUMPLIMIENTO_PCT'].mean().round(2)

# Crear figura con scatter plot
fig, ax = plt.subplots(figsize=(12, 7))

# Scatter plot: SALDO vs DISCREPANCIA, coloreado por GRUPO
grupos = df_op['GRUPO'].unique()
colores = plt.cm.Set3(range(len(grupos)))

for idx, grupo in enumerate(grupos):
    mask = df_op['GRUPO'] == grupo
    ax.scatter(
        df_op[mask]['SALDO'],
        df_op[mask]['DISCREPANCIA'],
        alpha=0.6,
        s=80,
        label=grupo,
        color=colores[idx],
        edgecolors='black',
        linewidth=0.5
    )

# Linea de referencia en y=0 (sin discrepancia)
ax.axhline(y=0, color='red', linestyle='--', linewidth=2, label='Sin discrepancia', alpha=0.7)
ax.axvline(x=0, color='gray', linestyle=':', linewidth=1, alpha=0.5)

ax.set_xlabel('Saldo (CANTIDAD - CANTIDAD_DESP) [unidades]', fontsize=11, fontweight='bold')
ax.set_ylabel('Discrepancia (CANTIDAD - CANTIDAD_DESP) [unidades]', fontsize=11, fontweight='bold')
ax.set_title('Relacion entre Saldo y Discrepancia por Grupo de Producto\nSeccion 5: Discrepancias Operacionales', 
             fontsize=13, fontweight='bold', pad=20)
ax.legend(loc='best', fontsize=9, framealpha=0.9)
ax.grid(True, alpha=0.3, linestyle=':')

plt.tight_layout()
plt.savefig('img/grafico_5.png', bbox_inches='tight')
plt.show()

# Analisis por ESTADO1
print('\n--- ANALISIS DE DISCREPANCIAS POR ESTADO ---')
discrepancia_por_estado = df_op.groupby('ESTADO1').agg({
    'TIENE_DISCREPANCIA': ['sum', 'count'],
    'DISCREPANCIA': ['mean', 'sum'],
    'CUMPLIMIENTO_PCT': 'mean'
}).round(2)
discrepancia_por_estado.columns = ['Lineas_con_Discrepancia', 'Total_Lineas', 
                                    'Discrepancia_Promedio', 'Discrepancia_Total',
                                    'Cumplimiento_Promedio_PCT']
print(discrepancia_por_estado)

# Analisis por GRUPO
print('\n--- ANALISIS DE DISCREPANCIAS POR GRUPO ---')
discrepancia_por_grupo = df_op.groupby('GRUPO').agg({
    'TIENE_DISCREPANCIA': ['sum', 'count'],
    'DISCREPANCIA': ['mean', 'sum'],
    'CUMPLIMIENTO_PCT': 'mean'
}).round(2)
discrepancia_por_grupo.columns = ['Lineas_con_Discrepancia', 'Total_Lineas',
                                   'Discrepancia_Promedio', 'Discrepancia_Total',
                                   'Cumplimiento_Promedio_PCT']
print(discrepancia_por_grupo)

# Hallazgo principal
print(f'\nHallazgo Principal: {lineas_con_discrepancia} de {total_lineas} lineas ({porcentaje_discrepancia}%) presentan discrepancia entre solicitado y despachado, con cumplimiento promedio de {cumplimiento_promedio}%.')

## 6. Síntesis Ejecutiva y Recomendaciones

Esta sección consolida los hallazgos principales del análisis de ventas de Ecovida (Alimentos Claudet), cuantificando su impacto financiero y traduciendo los insights en recomendaciones operacionales priorizadas. El objetivo es proporcionar a la dirección una visión clara de las oportunidades críticas y las acciones necesarias para optimizar la rentabilidad y el desempeño comercial en el corto y mediano plazo.

In [ ]:
# SECCION 6: SINTESIS EJECUTIVA Y RECOMENDACIONES
# Consolidacion de hallazgos y recomendaciones accionables

import pandas as pd
import numpy as np
from datetime import datetime

# ============================================================================
# 1. CALCULO DE METRICAS CLAVE PARA LOS 5 HALLAZGOS PRINCIPALES
# ============================================================================

# Hallazgo 1: Desempeño por canal
desempeño_canal = df_op.groupby('CANAL').agg({
    'Neto': ['sum', 'count']
})

---

## Conclusiones

Este análisis respondió las siguientes preguntas de negocio:

- ¿Cuál es el porcentaje de quiebre (saldo > 0) en los documentos y cómo se distribuye entre ESTADO1 y ESTADO2?
- ¿Qué productos tienen mayor incidencia de quiebre de saldo y cuál es su impacto financiero (Valor_Saldo)?
- ¿Cuáles son las diferencias en patrones de quiebre entre documentos en ESTADO11 vs ESTADO20 (o equivalentes)?
- ¿Cuántas líneas de documento tienen discrepancia entre CANTIDAD y CANTIDAD_DESP, y qué representa esto en términos de cumplimiento operacional?
- ¿Cómo ha evolucionado la tasa de quiebre de saldo a lo largo del período analizado?

---
*Análisis generado con Python · pandas · matplotlib · seaborn*
*Dataset: 86,932 transacciones · 23 de marzo de 2021 - 13 de enero de 2025*